#02_silver_contratos.sql

Este notebook serve para transformar dados da tabela bronze.contratos_planos (raw/string) em uma tabela silver valida tipada e deduplicada: silver.fact_contratos.

##Objetos

- tipar campos (timestamp, decimal)
- padronizar textos (trim/upper)
- deduplicar por contrato_id mantendo o registro mais recente (ingestion_ts)
- aplicar regras de qualidade e separar rejects com motivo (reject_reason)
- persistir de forma idempotente via MERGE (delta) usando row_hash
- processar incrementalmente usando checkpoint(silver_ops.pipeline_checkpoint)

##Estratégia:
- SQL-first com TEMP VIEWs (staging)
- TRY_CAST / TRY_TO_TIMESTAMP para conversões resilientes
- row_number() para dedupe determinístico
- rejects em tabela dedicada
- MERGE com row_hash

##Chave natural:
contrato_id

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_rejects;
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Pipeline checkpoint

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.pipeline_checkpoint (
  table_name STRING,
  last_ingestion_ts TIMESTAMP,
  last_run_ts TIMESTAMP,
  status STRING
)
USING DELTA;

##Tabelas destino (silver + rejects)

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver.fact_contratos (
  contrato_id STRING,
  beneficiario_id STRING,
  tipo_plano STRING,
  valor_mensal DECIMAL(10,2),
  coparticipacao_flag STRING,
  data_inicio_vigencia TIMESTAMP,
  data_fim_vigencia TIMESTAMP,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  row_hash STRING
)
USING DELTA;

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_rejects.fact_contratos (
  contrato_id STRING,
  beneficiario_id STRING,
  tipo_plano STRING,
  valor_mensal STRING,
  coparticipacao_flag STRING,
  data_inicio_vigencia STRING,
  data_fim_vigencia STRING,
  ingestion_ts TIMESTAMP,
  load_id STRING,
  source_file STRING,
  reject_reason STRING,
  reject_ts TIMESTAMP
)
USING DELTA;

##Leitura incremental do Bronze

In [0]:
CREATE OR REPLACE TEMP VIEW stg_contratos_raw AS
WITH last_cp AS (
  SELECT coalesce(max(last_ingestion_ts), timestamp('1900-01-01')) AS last_ingestion_ts
  FROM healthcare_dev.silver_ops.pipeline_checkpoint
  WHERE table_name = 'fact_contratos'
)
SELECT *
FROM healthcare_dev.bronze.contratos_planos
WHERE ingestion_ts >= (SELECT last_ingestion_ts FROM last_cp) - INTERVAL 3 DAYS;

##Tipagem e Padronização

In [0]:
CREATE OR REPLACE TEMP VIEW stg_contratos_typed AS
SELECT
  cast(contrato_id as string) AS contrato_id,
  cast(beneficiario_id as string) AS beneficiario_id,
  upper(trim(cast(tipo_plano as string))) AS tipo_plano,
  try_cast(valor_mensal as decimal(10,2)) AS valor_mensal,
  upper(trim(cast(coparticipacao_flag as string))) AS coparticipacao_flag,
  try_to_timestamp(data_inicio_vigencia) AS data_inicio_vigencia,
  try_to_timestamp(data_fim_vigencia) AS data_fim_vigencia,
  ingestion_ts,
  load_id,
  source_file
FROM stg_contratos_raw;

##Deduplicação determinística (contrato_id)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_contratos_dedup AS
SELECT *
FROM (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY contrato_id
      ORDER BY ingestion_ts DESC
    ) AS rn
  FROM stg_contratos_typed
) x
WHERE rn = 1;

##Regras de qualidade (classificação + motivo)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_contratos_classified AS
SELECT
  *,
  CASE
    WHEN contrato_id IS NULL OR contrato_id = '' THEN 'MISSING_CONTRATO_ID'
    WHEN beneficiario_id IS NULL OR beneficiario_id = '' THEN 'MISSING_BENEFICIARIO_ID'
    WHEN data_inicio_vigencia IS NULL THEN 'INVALID_DATA_INICIO_VIGENCIA'
    WHEN data_fim_vigencia IS NOT NULL AND data_fim_vigencia < data_inicio_vigencia THEN 'DATA_FIM_LT_INICIO'
    WHEN valor_mensal IS NOT NULL AND valor_mensal < 0 THEN 'NEGATIVE_VALOR_MENSAL'
    ELSE NULL
  END AS reject_reason
FROM stg_contratos_dedup;

##Persiste rejects

In [0]:
INSERT INTO healthcare_dev.silver_rejects.fact_contratos
SELECT
  contrato_id,
  beneficiario_id,
  tipo_plano,
  cast(valor_mensal as string) AS valor_mensal,
  coparticipacao_flag,
  cast(data_inicio_vigencia as string) AS data_inicio_vigencia,
  cast(data_fim_vigencia as string) AS data_fim_vigencia,
  ingestion_ts,
  load_id,
  source_file,
  reject_reason,
  current_timestamp() AS reject_ts
FROM stg_contratos_classified
WHERE reject_reason IS NOT NULL;

##Valid + row_hash (para MERGE idempotente)

In [0]:
CREATE OR REPLACE TEMP VIEW stg_contratos_valid AS
SELECT
  contrato_id,
  beneficiario_id,
  tipo_plano,
  valor_mensal,
  coparticipacao_flag,
  data_inicio_vigencia,
  data_fim_vigencia,
  ingestion_ts,
  load_id,
  source_file,
  sha2(concat_ws('||',
    contrato_id,
    coalesce(beneficiario_id,''),
    coalesce(tipo_plano,''),
    coalesce(cast(valor_mensal as string),''),
    coalesce(coparticipacao_flag,''),
    coalesce(cast(data_inicio_vigencia as string),''),
    coalesce(cast(data_fim_vigencia as string),'')
  ), 256) AS row_hash
FROM stg_contratos_classified
WHERE reject_reason IS NULL;

##Merge na Silver

In [0]:
MERGE INTO healthcare_dev.silver.fact_contratos AS tgt
USING stg_contratos_valid AS src
ON tgt.contrato_id = src.contrato_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

##Atualiza checkpoint

In [0]:
INSERT INTO healthcare_dev.silver_ops.pipeline_checkpoint
SELECT
  'fact_contratos' AS table_name,
  max(ingestion_ts) AS last_ingestion_ts,
  current_timestamp() AS last_run_ts,
  'SUCCESS' AS status
FROM stg_contratos_valid;

##Checa sanidade

In [0]:
SELECT COUNT(*) AS n_silver_contratos
FROM healthcare_dev.silver.fact_contratos;

In [0]:
SELECT COUNT(*) AS n_rejects_contratos
FROM healthcare_dev.silver_rejects.fact_contratos;

In [0]:
--valida regra data_fim_lt_inicio
SELECT reject_reason, COUNT(*) AS qtd
FROM healthcare_dev.silver_rejects.fact_contratos
GROUP BY reject_reason
ORDER BY qtd DESC;

In [0]:
SELECT contrato_id, data_inicio_vigencia, data_fim_vigencia, reject_reason
FROM healthcare_dev.silver_rejects.fact_contratos
WHERE reject_reason = 'DATA_FIM_LT_INICIO'
LIMIT 20;